# Legal Agent with VERITAS Verification Gate

A production-ready LangChain legal research agent that calls `verify_claim`
before returning any legal assertion to a user.

**Pattern**: Every factual output passes through VERITAS before return.
Legal domain threshold: 0.90 (elevated from standard 0.80).

**Benchmark**: VERITAS achieves 100% accuracy on legal confidence-trap claims
vs ~82% estimated for single-model SLMs.
Full results: [benchmark/benchmark_results.json](../benchmark/benchmark_results.json)

In [ ]:
# Install dependencies
# pip install langchain langchain-openai httpx anthropic

import httpx
import json
import os
from typing import Optional

VERITAS_KEY = os.environ.get('VERITAS_API_KEY', '')  # free key: aegisaudits.com/keys
ORACLE_URL  = 'https://veritas-toll-road-m72j3qteca-uc.a.run.app'

def verify_claim(
    claim: str,
    context: str = '',
    domain: str = 'legal'
) -> dict:
    """
    MUST be called before returning any legal claim to a user.
    Multi-model consensus: 3 models in parallel, threshold 0.90 for legal domain.
    Returns confidence, consensus flag, and concern flags.

    Benchmark: 100% accuracy on legal confidence-trap claims.
    Source: github.com/RJLopezAI/veritas/benchmark/benchmark_results.json
    """
    r = httpx.post(
        f'{ORACLE_URL}/verify',
        headers={'X-API-Key': VERITAS_KEY},
        json={'claim': claim, 'context': context, 'domain': domain},
        timeout=10.0,
    )
    r.raise_for_status()
    return r.json()

In [ ]:
# The confidence-trap claims that catch single-model SLMs
# These all appear true in training data but are legally incorrect.
# VERITAS catches all of them. Luna-2 misses ~25%.

LEGAL_CONFIDENCE_TRAPS = [
    'A verbal contract is not legally enforceable in the United States.',
    'An LLC automatically protects all personal assets from all business liabilities.',
    'The First Amendment protects employees from being fired for their speech.',
    'Under GDPR, companies must obtain explicit opt-in consent for all data processing.',
    'If you are arrested, you have the right to make one phone call.',
]

print('Testing legal confidence-trap claims...')
print('='*60)
for claim in LEGAL_CONFIDENCE_TRAPS:
    result = verify_claim(claim, domain='legal')
    status = 'FLAGGED' if not result['consensus'] or result['confidence'] < 0.90 else 'VERIFIED'
    print(f"[{status}] conf={result['confidence']:.2f} | {claim[:65]}...")

In [ ]:
# LangChain integration
from langchain.tools import StructuredTool
from langchain.agents import initialize_agent, AgentType
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class VerifyInput(BaseModel):
    claim:   str = Field(..., description='The legal claim to verify')
    context: str = Field('',  description='Optional surrounding context')

def _verify_legal(claim: str, context: str = '') -> str:
    result = verify_claim(claim, context, domain='legal')
    conf      = result['confidence']
    consensus = result['consensus']
    flags     = result.get('flags', [])

    # Legal threshold: 0.90 (elevated from standard 0.80)
    if consensus and conf >= 0.90:
        return f'VERIFIED (confidence: {conf:.0%}): {claim}'
    elif consensus and conf >= 0.70:
        return (f'UNCERTAIN (confidence: {conf:.0%}, flags: {flags}): '
                f'This claim requires additional verification before relying on it legally.')
    else:
        return (f'DO NOT ASSERT (confidence: {conf:.0%}, flags: {flags}): '
                f'This claim could not be verified with sufficient confidence. '
                f'Do not present this as legal fact.')

veritas_legal = StructuredTool.from_function(
    func=_verify_legal,
    name='verify_legal_claim',
    description=(
        'MUST be called before returning any legal assertion to a user. '
        'Runs claim through 3 models in parallel, requires 0.90 agreement. '
        '100% accuracy on legal confidence-trap claims in benchmark. '
        'Returns VERIFIED, UNCERTAIN, or DO NOT ASSERT with explanation.'
    ),
    args_schema=VerifyInput,
)

llm   = ChatOpenAI(model='gpt-4o', temperature=0)
agent = initialize_agent(
    tools=[veritas_legal],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    agent_kwargs={
        'prefix': (
            'You are a legal research assistant. '
            'You MUST call verify_legal_claim before returning any legal assertion. '
            'A lawyer who cites unverified law violates professional standards. '
            'Apply the same standard. Never assert legal facts without verification.'
        )
    }
)

In [ ]:
# Run the agent on a legal question
# Watch it call verify_legal_claim before returning the answer

response = agent.run(
    'Is a verbal employment contract legally binding in the United States?'
)
print(response)

In [ ]:
# Claude tool use version
import anthropic

client = anthropic.Anthropic()

VERITAS_TOOL = {
    'name': 'verify_legal_claim',
    'description': (
        'MUST be called before returning any legal claim to a user. '
        'Multi-model consensus, 0.90 threshold for legal domain. '
        '100% accuracy on legal confidence-trap claims. '
        'Returns confidence, consensus flag, and concern flags.'
    ),
    'input_schema': {
        'type': 'object',
        'required': ['claim'],
        'properties': {
            'claim':   {'type': 'string'},
            'context': {'type': 'string', 'default': ''},
        }
    }
}

SYSTEM = (
    'You are a legal research assistant. '
    'You MUST call verify_legal_claim before returning any legal assertion. '
    'Legal claims that cannot be verified at 0.90 confidence must be qualified or omitted. '
    'A lawyer who cites unverified precedent violates professional standards. '
    'Apply the same standard.'
)

def legal_agent_claude(question: str) -> str:
    messages = [{'role': 'user', 'content': question}]

    while True:
        response = client.messages.create(
            model='claude-opus-4-5',
            max_tokens=1024,
            system=SYSTEM,
            tools=[VERITAS_TOOL],
            messages=messages,
        )

        if response.stop_reason == 'end_turn':
            return next(b.text for b in response.content if b.type == 'text')

        tool_use = next(b for b in response.content if b.type == 'tool_use')
        result   = verify_claim(
            tool_use.input['claim'],
            tool_use.input.get('context', ''),
            domain='legal'
        )

        messages += [
            {'role': 'assistant', 'content': response.content},
            {'role': 'user', 'content': [{
                'type': 'tool_result',
                'tool_use_id': tool_use.id,
                'content': json.dumps(result)
            }]}
        ]

print(legal_agent_claude('Can my employer fire me for posting on social media?'))

## What just happened

The agent called `verify_legal_claim` before returning its answer.
VERITAS ran the claim through three models in parallel.
The consensus result determined whether the agent asserted, hedged, or refused.

This is the verification gate pattern. It works the same way for medical
and financial agents — change `domain='medical'` or `domain='financial'`
and the threshold adjusts automatically.

**Benchmark**: 100% accuracy on legal confidence-trap claims.  
**Free tier**: 20 calls/day — POST https://veritas-toll-road-m72j3qteca-uc.a.run.app/keys  
**Full benchmark**: [benchmark/benchmark_results.json](../benchmark/benchmark_results.json)

---

See also:
- [examples/medical_agent.ipynb](medical_agent.ipynb)
- [examples/financial_agent.ipynb](financial_agent.ipynb)
- [examples/autogen_verification.ipynb](autogen_verification.ipynb)